In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path
import os
import requests
import pandas as pd
import time
from pathlib import Path

In [2]:


API_KEY = os.environ["TIINGO_API_KEY"]

ticker = "AAPL"

url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"

params = {
    "startDate": "2010-01-01",
    "endDate": "2026-07-31",
    "token": API_KEY
}

response = requests.get(url, params=params)
response.raise_for_status()

aapl = pd.DataFrame(response.json())

aapl.head()

KeyError: 'TIINGO_API_KEY'

In [ ]:
TICKERS = [
    "AAPL", "AMGN", "AMZN", "AXP", "BA",
    "CAT", "CRM", "CSCO", "CVX", "DIS",
    "GOOGL", "GS", "HD", "HON", "IBM",
    "JNJ", "JPM", "KO", "MCD", "MMM",
    "MRK", "MSFT", "NKE", "NVDA", "PG",
    "SHW", "TRV", "UNH", "V", "WMT"
]

print("Number of assets:", len(TICKERS))

In [ ]:

RAW_DIR = Path("../data/raw/tiingo")
RAW_DIR.mkdir(parents=True, exist_ok=True)

prices = {}
failed_tickers = []

for ticker in TICKERS:
    print(f"Downloading {ticker} ...")

    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"

    params = {
        "startDate": "2010-01-01",
        "endDate": "2026-07-31",
        "token": API_KEY
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        df = pd.DataFrame(response.json())

        if df.empty:
            print(f"  WARNING: {ticker} returned no data")
            failed_tickers.append(ticker)
            continue

        # Save the complete API response without manual modification
        df.to_csv(
            RAW_DIR / f"{ticker}_tiingo_eod.csv",
            index=False
        )

        # Convert date for construction of the price matrix
        df["date"] = pd.to_datetime(df["date"], utc=True)
        df = df.set_index("date").sort_index()

        prices[ticker] = df["adjClose"]

        print(f"  OK: {len(df)} observations")

    except Exception as e:
        print(f"  FAILED: {ticker}: {e}")
        failed_tickers.append(ticker)

    time.sleep(1)

In [ ]:
print("Successful downloads:", len(prices))
print("Failed downloads:", failed_tickers)

In [ ]:
adj_close = pd.DataFrame(prices)

print("Shape:", adj_close.shape)
display(adj_close.head())

In [ ]:
adj_close.to_csv("../data/raw/djia30_tiingo_adjclose_2010_2026.csv")

In [ ]:
RAW_FILE = Path("../data/raw/djia30_tiingo_adjclose_2010_2026.csv")

prices = pd.read_csv(
    RAW_FILE,
    index_col=0,
    parse_dates=True
)

print("Shape:", prices.shape)
print("First date:", prices.index.min())
print("Last date:", prices.index.max())

prices.head()

In [ ]:
missing_count = prices.isna().sum()

missing_pct = prices.isna().mean() * 100

first_valid_date = prices.apply(
    lambda column: column.first_valid_index()
)

last_valid_date = prices.apply(
    lambda column: column.last_valid_index()
)

In [ ]:
print(missing_count)
print(missing_pct)

In [ ]:
returns = prices / prices.shift(1) - 1 
returns = returns.iloc[1:].copy()
print("Returns shape:", returns.shape)
print("Total NaNs:", returns.isna().sum().sum())
print("Number of infinite values:", np.isinf(returns.to_numpy()).sum())

display(returns.head())
display(returns.describe().T)


In [ ]:
#duplicates test
duplicate_dates = returns.index.duplicated().sum()

print("Duplicate dates:", duplicate_dates)

In [ ]:
#Date order test
print(
    "Dates sorted:",
    returns.index.is_monotonic_increasing
)

In [ ]:
extreme_table = (
    returns
    .stack()
    .rename("return")
    .reset_index()
)

extreme_table.columns = [
    "date",
    "ticker",
    "return"
]

extreme_table["abs_return"] = (
    extreme_table["return"].abs()
)

extreme_table = extreme_table.sort_values(
    "abs_return",
    ascending=False
)

display(extreme_table.head(20))

In [ ]:
large_moves = (
    extreme_table[
        extreme_table["abs_return"] >= 0.20
    ]
)

print(
    "Observations with |daily return| >= 20%:",
    len(large_moves)
)

display(large_moves)

In [ ]:
return_range = pd.DataFrame({
    "min_return": returns.min(),
    "max_return": returns.max()
})

return_range["max_abs_return"] = (
    returns.abs().max()
)

return_range = return_range.sort_values(
    "max_abs_return",
    ascending=False
)

display(return_range)

In [ ]:
zero_mask = np.isclose(
    returns.to_numpy(),
    0.0,
    atol=1e-12
)

zero_counts = pd.Series(
    zero_mask.sum(axis=0),
    index=returns.columns,
    name="zero_return_days"
)

zero_counts.sort_values(ascending=False)

In [ ]:
def longest_zero_run(series):
    zero = np.isclose(series.to_numpy(), 0.0, atol=1e-12)

    longest = 0
    current = 0

    for is_zero in zero:
        if is_zero:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest


longest_zero_runs = returns.apply(longest_zero_run)

zero_summary = pd.DataFrame({
    "zero_return_days": zero_counts,
    "longest_zero_run": longest_zero_runs
})

zero_summary = zero_summary.sort_values(
    "longest_zero_run",
    ascending=False
)

display(zero_summary)

In [ ]:
import matplotlib.pyplot as plt

all_returns = returns.to_numpy().flatten()

plt.figure(figsize=(8, 5))
plt.hist(all_returns, bins=100)

plt.xlabel("Daily return")
plt.ylabel("Frequency")
plt.title("Distribution of Daily Returns")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    all_returns[
        (all_returns >= -0.10) &
        (all_returns <= 0.10)
    ],
    bins=100
)

plt.xlabel("Daily return")
plt.ylabel("Frequency")
plt.title("Distribution of Daily Returns (-10% to +10%)")

plt.show()

In [ ]:
selected = ["AAPL", "NVDA", "BA", "KO"]

for ticker in selected:
    plt.figure(figsize=(9, 4))

    plt.plot(prices.index, prices[ticker])

    plt.xlabel("Date")
    plt.ylabel("Adjusted Close")
    plt.title(f"{ticker} Adjusted Close")

    plt.show()

In [ ]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RETURNS_FILE = PROCESSED_DIR / "returns.csv"

returns.to_csv(RETURNS_FILE)

print("Saved to:", RETURNS_FILE.resolve())

In [ ]:
returns_loaded = pd.read_csv(
    RETURNS_FILE,
    index_col=0,
    parse_dates=True
)

print("Loaded shape:", returns_loaded.shape)
print("NaNs:", returns_loaded.isna().sum().sum())

